#  RNN Language Model

## 핵심 개념
| 개념 | 설명 |
|------|------|
| **Language Model** | $P(w_t \mid w_1, \dots, w_{t-1})$ — 이전 단어들이 주어졌을 때 다음 단어 확률 |
| **Teacher Forcing** | 학습 시 정답 토큰을 다음 입력으로 사용 |
| **Truncated BPTT** | 배치 간 gradient 흐름을 `detach()`로 차단 |
| **Gradient Clipping** | gradient norm 상한을 두어 폭발 방지 |
| **Weight Tying** | Embedding ↔ FC 가중치 공유 (파라미터↓, 성능↑) |
| **Perplexity** | $\exp(\text{CrossEntropy})$ — LM 평가 지표, 낮을수록 좋음 |

```
[입력]  BOS  w1   w2   w3  ...
[출력]  w1   w2   w3   EOS ...
         ↑    ↑    ↑    ↑
       각 시점에서 다음 단어 예측
```

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import math

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


## 1️⃣ 어휘 사전 (Vocabulary)

In [2]:
class Vocabulary:
    PAD_TOKEN = '<PAD>'
    UNK_TOKEN = '<UNK>'
    BOS_TOKEN = '<BOS>'
    EOS_TOKEN = '<EOS>'

    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        for tok in [self.PAD_TOKEN, self.UNK_TOKEN, self.BOS_TOKEN, self.EOS_TOKEN]:
            self._add(tok)

    def _add(self, word):
        if word not in self.word2idx:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def build(self, sentences, min_freq=1):
        from collections import Counter
        freq = Counter(w for sent in sentences for w in sent)
        for word, cnt in freq.items():
            if cnt >= min_freq:
                self._add(word)
        print(f'[Vocab] 크기: {len(self)} (특수토큰 4개 포함)')

    def encode(self, words):
        unk = self.word2idx[self.UNK_TOKEN]
        return [self.word2idx.get(w, unk) for w in words]

    def decode(self, indices):
        return [self.idx2word.get(i, self.UNK_TOKEN) for i in indices]

    def __len__(self): return len(self.word2idx)

## 2️⃣ 데이터셋

입력: `[BOS, w1, w2, ..., w_{n-1}]`  
출력: `[w1, w2, ..., w_{n-1}, EOS]`

In [3]:
class LMDataset(Dataset):
    def __init__(self, sentences, vocab, max_len=30):
        self.data = []
        bos = vocab.word2idx[Vocabulary.BOS_TOKEN]
        eos = vocab.word2idx[Vocabulary.EOS_TOKEN]
        for sent in sentences:
            ids = vocab.encode(sent[:max_len])
            self.data.append(([bos] + ids, ids + [eos]))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


def collate_fn(batch, pad_idx):
    inputs, targets = zip(*batch)
    max_len = max(len(s) for s in inputs)
    pad_inp, pad_tgt = [], []
    for inp, tgt in zip(inputs, targets):
        diff = max_len - len(inp)
        pad_inp.append(inp + [pad_idx] * diff)
        pad_tgt.append(tgt + [pad_idx] * diff)
    return (torch.tensor(pad_inp, dtype=torch.long),
            torch.tensor(pad_tgt, dtype=torch.long))

## 3️⃣ RNN Language Model

```
Embedding → LSTM (다층) → Dropout → Linear → Softmax
```

**RNN 타입 선택:**
- `RNN`  : 가장 단순, gradient 소실 심함
- `GRU`  : 게이트 2개, LSTM보다 파라미터 적음
- `LSTM` : 게이트 4개, Cell state로 장기 의존성 처리

In [4]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64,
                 num_layers=2, dropout=0.3, rnn_type='LSTM',
                 tie_weights=True, pad_idx=0):
        super().__init__()
        self.rnn_type   = rnn_type
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_drop = nn.Dropout(dropout)

        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(embed_dim, hidden_dim, num_layers,
                           batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)

        self.out_drop = nn.Dropout(dropout)
        self.fc       = nn.Linear(hidden_dim, vocab_size)

        # ★ Weight Tying: Embedding ↔ FC 가중치 공유
        if tie_weights and embed_dim == hidden_dim:
            self.fc.weight = self.embedding.weight
            print('[Model] Weight tying 적용됨')

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        for name, p in self.rnn.named_parameters():
            if 'weight' in name: nn.init.orthogonal_(p)
            elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, x, hidden=None):
        emb    = self.embed_drop(self.embedding(x))   # (B, T, E)
        out, hidden = self.rnn(emb, hidden)            # (B, T, H)
        logits = self.fc(self.out_drop(out))           # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        zeros = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        return (zeros, zeros.clone()) if self.rnn_type == 'LSTM' else zeros

    def detach_hidden(self, hidden):
        """Truncated BPTT: 이전 배치 gradient 차단"""
        if isinstance(hidden, tuple):
            return tuple(h.detach() for h in hidden)
        return hidden.detach()

## 4️⃣ 학습 & 평가 함수

In [5]:
def train_epoch(model, loader, optimizer, criterion, device, clip=1.0):
    model.train()
    total_loss, total_tokens = 0.0, 0
    pad_idx = criterion.ignore_index

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        hidden = model.init_hidden(inputs.size(0), device)
        optimizer.zero_grad()

        logits, hidden = model(inputs, hidden)
        loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
        loss.backward()

        # ★ Gradient Clipping
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        n_tokens      = (targets != pad_idx).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens

    avg = total_loss / total_tokens
    return avg, math.exp(avg)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    pad_idx = criterion.ignore_index
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        hidden = model.init_hidden(inputs.size(0), device)
        logits, _ = model(inputs, hidden)
        loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
        n_tokens      = (targets != pad_idx).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens
    avg = total_loss / total_tokens
    return avg, math.exp(avg)

## 5️⃣ 텍스트 생성 (Temperature Sampling)

In [6]:
@torch.no_grad()
def generate(model, vocab, seed_text, max_len=20, temperature=1.0, device='cpu'):
    """
    temperature = 0   → Greedy decoding
    temperature < 1   → 보수적 (분포 뾰족)
    temperature > 1   → 창의적 (분포 평탄)
    """
    model.eval()
    words  = seed_text.lower().split()
    ids    = [vocab.word2idx[Vocabulary.BOS_TOKEN]] + vocab.encode(words)
    input_t = torch.tensor([ids], device=device)
    hidden  = model.init_hidden(1, device)
    _, hidden = model(input_t, hidden)

    result, cur_id = words[:], ids[-1]
    eos_id = vocab.word2idx[Vocabulary.EOS_TOKEN]

    for _ in range(max_len):
        inp = torch.tensor([[cur_id]], device=device)
        logits, hidden = model(inp, hidden)
        logits = logits.squeeze()
        if temperature == 0:
            cur_id = logits.argmax().item()
        else:
            probs  = torch.softmax(logits / temperature, dim=-1)
            cur_id = torch.multinomial(probs, 1).item()
        if cur_id == eos_id: break
        result.append(vocab.idx2word.get(cur_id, Vocabulary.UNK_TOKEN))
    return ' '.join(result)

## 6️⃣ 데이터 준비 & 학습 실행

In [7]:
CORPUS = """
the cat sat on the mat
the dog ran in the park
a bird flew over the tree
the sun rises in the east
the moon shines at night
she reads a book every day
he loves to play the guitar
they walked along the river
the rain fell on the roof
a child laughed in the yard
the wind blew through the leaves
stars twinkle in the dark sky
the fire crackled in the hearth
waves crash upon the sandy shore
time flies when you are having fun
the teacher explained the lesson clearly
students listened carefully to the lecture
knowledge grows through practice and study
language models learn from text data
deep learning enables powerful representations
""".strip().split('\n')

tokenized    = [s.lower().split() for s in CORPUS]
split        = int(len(tokenized) * 0.8)
train_sents  = tokenized[:split]
valid_sents  = tokenized[split:]

vocab = Vocabulary()
vocab.build(train_sents)

pad_idx  = vocab.word2idx[Vocabulary.PAD_TOKEN]
collate  = lambda b: collate_fn(b, pad_idx)

train_loader = DataLoader(LMDataset(train_sents, vocab), batch_size=4, shuffle=True,  collate_fn=collate)
valid_loader = DataLoader(LMDataset(valid_sents, vocab), batch_size=4, shuffle=False, collate_fn=collate)
print(f'Train: {len(train_sents)}문장 / Valid: {len(valid_sents)}문장')

[Vocab] 크기: 72 (특수토큰 4개 포함)
Train: 16문장 / Valid: 4문장


In [8]:
model = RNNLanguageModel(
    vocab_size  = len(vocab),
    embed_dim   = 64,
    hidden_dim  = 64,
    num_layers  = 2,
    dropout     = 0.3,
    rnn_type    = 'LSTM',   # 'RNN' / 'GRU' / 'LSTM'
    tie_weights = True,
    pad_idx     = pad_idx,
).to(device)

print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(model)

[Model] Weight tying 적용됨
파라미터 수: 71,240
RNNLanguageModel(
  (embedding): Embedding(72, 64, padding_idx=0)
  (embed_drop): Dropout(p=0.3, inplace=False)
  (rnn): LSTM(64, 64, num_layers=2, batch_first=True, dropout=0.3)
  (out_drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=72, bias=True)
)


In [9]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val_ppl = float('inf')
print(f'{'Epoch':>6} | {'Train Loss':>10} | {'Train PPL':>10} | {'Val Loss':>9} | {'Val PPL':>8}')
print('=' * 55)

for epoch in range(1, 51):
    tr_loss, tr_ppl = train_epoch(model, train_loader, optimizer, criterion, device)
    vl_loss, vl_ppl = evaluate(model, valid_loader, criterion, device)
    scheduler.step(vl_loss)

    if vl_ppl < best_val_ppl:
        best_val_ppl = vl_ppl
        torch.save(model.state_dict(), '/tmp/best_rnnlm.pt')

    if epoch % 10 == 0 or epoch == 1:
        mark = ' ★' if vl_ppl == best_val_ppl else ''
        print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_ppl:>10.2f} | {vl_loss:>9.4f} | {vl_ppl:>8.2f}{mark}')

print(f'\n최고 Validation Perplexity: {best_val_ppl:.2f}')

 Epoch | Train Loss |  Train PPL |  Val Loss |  Val PPL
     1 |     4.3040 |      74.00 |    4.3671 |    78.81 ★
    10 |     3.9321 |      51.02 |    4.3470 |    77.24 ★
    20 |     3.5750 |      35.70 |    4.8417 |   126.68
    30 |     3.5659 |      35.37 |    4.8216 |   124.16
    40 |     3.5952 |      36.42 |    4.8225 |   124.28
    50 |     3.5642 |      35.31 |    4.8227 |   124.30

최고 Validation Perplexity: 77.24


## 7️⃣ 텍스트 생성 실험

In [10]:
model.load_state_dict(torch.load('/tmp/best_rnnlm.pt', map_location=device))

seeds = ['the cat', 'deep learning', 'the sun', 'a bird']
print('[텍스트 생성 결과]\n')
for seed in seeds:
    print(f'  Seed: "{seed}"')
    for temp in [0, 0.7, 1.2]:
        label = 'greedy' if temp == 0 else f'temp={temp}'
        gen   = generate(model, vocab, seed, max_len=12, temperature=temp, device=device)
        print(f'    [{label:>8}] {gen}')
    print()

[텍스트 생성 결과]

  Seed: "the cat"
    [  greedy] the cat the the the the the the the the the the the the
    [temp=0.7] the cat crash night
    [temp=1.2] the cat book night sky the in shines the teacher crash shore

  Seed: "deep learning"
    [  greedy] deep learning the the the the the the the the the the the the
    [temp=0.7] deep learning flew night along
    [temp=1.2] deep learning you ran play at having night

  Seed: "the sun"
    [  greedy] the sun the the the the the the the the the the the the
    [temp=0.7] the sun over play she in the the she
    [temp=1.2] the sun guitar day blew day to

  Seed: "a bird"
    [  greedy] a bird the the the the the the the the the the the the
    [temp=0.7] a bird a sun fun the
    [temp=1.2] a bird walked he park fell fun the moon



## 8️⃣ RNN 타입 비교 실험

RNN / GRU / LSTM 성능을 비교해봅니다.

In [11]:
results = {}
for rnn_type in ['RNN', 'GRU', 'LSTM']:
    m = RNNLanguageModel(len(vocab), embed_dim=64, hidden_dim=64,
                         num_layers=2, dropout=0.3,
                         rnn_type=rnn_type, tie_weights=True, pad_idx=pad_idx).to(device)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss(ignore_index=pad_idx)
    best_ppl = float('inf')
    for _ in range(40):
        train_epoch(m, train_loader, opt, crit, device)
        _, vl_ppl = evaluate(m, valid_loader, crit, device)
        best_ppl = min(best_ppl, vl_ppl)
    results[rnn_type] = best_ppl
    print(f'{rnn_type:>5}: Best Val PPL = {best_ppl:.2f}')

best_type = min(results, key=results.get)
print(f'\n★ 최고 성능 RNN 타입: {best_type} (PPL={results[best_type]:.2f})')

[Model] Weight tying 적용됨
  RNN: Best Val PPL = 66.55
[Model] Weight tying 적용됨
  GRU: Best Val PPL = 65.06
[Model] Weight tying 적용됨
 LSTM: Best Val PPL = 64.31

★ 최고 성능 RNN 타입: LSTM (PPL=64.31)
